# اكتشاف المعادن: مكتبة USGS Splib07 + صور الأقمار الصناعية
خطوات الدفتر: (1) تثبيت الحزمة (2) تنزيل مكتبة USGS وبناؤها (3) تحميل صورة قمر (4) المطابقة وعرض الخرائط (5) حفظ النتائج.

**مهم:** الصورة يجب أن تكون **انعكاسًا سطحيًا** (Sentinel-2 L2A مثلًا) لا قيم DN أو TOA.

In [ ]:
!pip -q install "git+https://github.com/yamenetoo/RremoteSensing.git#subdirectory=USGS" rasterio matplotlib tqdm

## 1) المكتبة الطيفية (تنزيل مرة واحدة وحفظها في Drive)

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

WORK = Path("/content/drive/MyDrive/splib07")   # يُحفظ هنا ولا يُنزَّل ثانية
WORK.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "splib07_vnir_swir.npz"

In [ ]:
from usgs_splib07 import SpectralLibrary, build_library, download_splib07

if CACHE.exists():
    lib = SpectralLibrary.load(CACHE)
else:
    root = download_splib07(WORK)        # من USGS ScienceBase (قد يستغرق دقائق)
    lib = build_library(root, library="splib07b", chapters="MSV",   # M معادن، S تربة، V نباتات
                        kinds=("AREF",), start=350, stop=2500, step=5)
    lib.save(CACHE)
print(lib)
lib.meta["material"].value_counts().head(15)

## 2) الصورة
اختر **واحدًا** من الخيارات:
- **A** رفع ملف GeoTIFF لديك.
- **B** تصدير Sentinel-2 L2A من Google Earth Engine.
- **C** مشهد اصطناعي للتجربة فقط (لا بيانات حقيقية).

In [ ]:
# ---------- A) ارفع ملف GeoTIFF ----------
from google.colab import files
up = files.upload()
IMG_PATH = next(iter(up))
# اضبط حسب بياناتك:
SENSOR = "sentinel2a"                                   # أو landsat8 / aster
BANDS  = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]  # ترتيب النطاقات في الملف
SCALE, OFFSET = 1e-4, 0.0                               # Sentinel-2 L2A. لاندسات C2 L2: 2.75e-5 و -0.2

In [ ]:
# ---------- B) Google Earth Engine → Drive (بدّل المنطقة والتاريخ) ----------
import ee
ee.Authenticate(); ee.Initialize(project="YOUR_GCP_PROJECT")

region = ee.Geometry.Rectangle([34.9, 29.4, 35.1, 29.6])   # [غرب, جنوب, شرق, شمال]
BANDS = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
img = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
       .filterBounds(region).filterDate("2024-01-01", "2024-12-31")
       .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
       .select(BANDS).median().clip(region))
# كل النطاقات بدقة 20 م حتى تتطابق الشبكة
task = ee.batch.Export.image.toDrive(image=img.toInt16(), description="s2_scene", folder="gee",
                                     fileNamePrefix="s2_scene", region=region, scale=20, crs="EPSG:4326",
                                     maxPixels=1e9)
task.start()
# انتظر انتهاء المهمة (راقبها في code.earthengine.google.com/tasks) ثم:
IMG_PATH = "/content/drive/MyDrive/gee/s2_scene.tif"
SENSOR, SCALE, OFFSET = "sentinel2a", 1e-4, 0.0

In [ ]:
# ---------- C) مشهد اصطناعي للتجربة ----------
from usgs_splib07.synthetic import demo_library, synthetic_scene
demo = demo_library(step=1.0)
image, truth, labels = synthetic_scene(demo, "sentinel2a", noise=0.005, seed=1)
lib = demo                 # يستبدل المكتبة الحقيقية في هذا الخيار فقط

## 3) قراءة الصورة (تخطَّ هذه الخلية في الخيار C)

In [ ]:
from usgs_splib07 import Image
image = Image.read(IMG_PATH, sensor=SENSOR, band_names=BANDS, scale=SCALE, offset=OFFSET, nodata=-9999)
print(image.n_bands, "bands,", image.rows, "x", image.cols)
# صورة فائقة الطيف (EnMAP/PRISMA/EMIT محوّلة إلى GeoTIFF)؟ إن كانت الأطوال في الوسوم يكفي:
# image = Image.read("scene.tif", scale=1e-4)    أو مرّرها: wavelengths=[...], fwhm=10

## 4) المطابقة

In [ ]:
from usgs_splib07 import match_image

res = match_image(image, lib,
                  group_by="material",     # أفضل عينة لكل مادة
                  max_angle_deg=10.0,      # أكبر زاوية طيفية مقبولة (أقل = أكثر صرامة)
                  mask_vegetation=0.4)     # يتجاهل البكسلات الخضراء (NDVI > 0.4)
res.counts().head(15)

In [ ]:
import matplotlib.pyplot as plt, numpy as np
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
red = [image.band_names.index(n) for n in ("B4","B3","B2")] if image.band_names and "B4" in image.band_names else [0,1,2]
rgb = np.clip(np.moveaxis(np.nan_to_num(image.data[red]), 0, -1) * 3.5, 0, 1)
ax[0].imshow(rgb); ax[0].set_title("RGB")
cm = np.ma.masked_less(res.class_map, 0)
im = ax[1].imshow(cm, cmap="tab20", interpolation="nearest"); ax[1].set_title("best-match material")
a = ax[2].imshow(res.angle_deg, cmap="viridis_r"); ax[2].set_title("spectral angle (deg, low = good)")
plt.colorbar(a, ax=ax[2], fraction=0.046)
for x in ax: x.set_xticks([]); x.set_yticks([])
plt.tight_layout(); plt.show()
# أسماء الفئات بالأرقام:
for i, name in enumerate(res.labels): print(i, name)

`angle_deg` صغيرة = تشابه جيد. `margin_deg` (الفرق بين أفضل وثاني أفضل مادة) صغير = نتيجة ملتبسة، فلا تثق بها.

In [ ]:
# نتيجة موثوقة فقط: زاوية صغيرة وهامش واضح
good = (res.angle_deg < 6) & (res.margin_deg > 1.0)
print(f"{good.mean():.1%} of pixels are confident matches")
plt.figure(figsize=(6,5)); plt.imshow(np.ma.masked_where(~good | (res.class_map < 0), res.class_map), cmap="tab20", interpolation="nearest")
plt.title("confident matches only"); plt.axis("off"); plt.show()

## 5) حفظ النتائج

In [ ]:
# GeoTIFF من 3 نطاقات (class, angle, margin) + ملف legend.csv. يحتاج صورة بإحداثيات (الخيار A/B).
out = res.to_geotiff("/content/drive/MyDrive/splib07/match.tif")
print(out)